# Create BMEL Awards (German Federal Ministry of Food and Agriculture)

Creates awards for the **Bundesministerium für Ernährung und Landwirtschaft (BMEL)** -- renamed
Bundesministerium für Landwirtschaft, Ernährung und Heimat (BMLEH) in 2025 -- from the federal
**Förderkatalog** (https://foerderportal.bund.de/foekat), using its own CSV export
("Ausgabe als Textdatei") of the detail search Ressort = BMLEH including completed projects.
The catalogue lists completed projects under the ministry responsible today, so BMVEL/BMELV/BMEL-era
projects are included. 8,633 hits, 1992-2026; projects administered by the BMEL project executing
agencies BLE (5,715) and FNR (2,916).

**Source choice.** The tracker pointed at FISA (fisaonline.de), a shared portal of several federal
and state funders. On 2026-10-01 every FISA project page returned HTTP 500/503 (home page up), and
the Förderkatalog is the ministry-level register already split by ministry, so no cross-funder
routing is needed (runbook §2.3.2): the script asserts Ressort = BMLEH on every row.

**Scope filter (script):** 2 bookkeeping rows dropped (a brochure payout "nur zum Zweck der
Auszahlung" and an EUR 0 "Sammelvorhabenbuchung" placeholder). Kept: research projects, model &
demonstration projects, the BMEL doctoral programme (DOKI/DOKA/DOKP, profiled "nicht FuE-relevante
Bildungsausgaben" by the catalogue but they are doctoral research projects), departmental research
contracts.

**Fields:** `funder_award_id` = FKZ (Förderkennzeichen, e.g. `2819107716`, `28DE103C22`,
`22003015`) -- the reference papers cite (529 of 1,319 distinct F4320323499 stub ids collapse).
`amount` = federal share in **EUR** (EUR 0 in the catalogue -> NULL). No PI names are published:
`lead_investigator` carries only the grantee institution (Förderempfänger) as affiliation, NULL
when the catalogue withholds it for data protection ("Keine Anzeige ..."). `funder_scheme` = the
Leistungsplansystematik text. No abstracts are published.

**Related funder rows (stubs that will not collapse, ids hash the funder):** BLE F4320335545
(284 of 619 distinct stub ids are FKZs in this ingest), FNR F4320326662 (117 of 397), BMVEL
(older name) F4320331007 (29 of 67).

**Prerequisites:** run `scripts/local/bmel_foekat_to_s3.py` (uploads
`s3://openalex-ingest/awards/bmel/bmel_projects.parquet`, §1.4 shrink guard).

**Funder (Path A, F4320*):** funder_id `4320323499` (Bundesministerium für Ernährung und
Landwirtschaft, ror 04jw21793, doi 10.13039/501100005908), read from `openalex.funders.funders`.

**Priority:** `532` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bmel_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/bmel/bmel_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects, COUNT(DISTINCT fkz) as distinct_fkz
FROM openalex.awards.bmel_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.bmel_raw;

In [ ]:
%sql
SELECT ressort, admin_agency, funding_profile, COUNT(*) AS n, ROUND(SUM(TRY_CAST(amount_eur AS DOUBLE)), 0) AS eur
FROM openalex.awards.bmel_raw
GROUP BY 1, 2, 3 ORDER BY n DESC;

## Step 1.6: Funder existence check (Path A)
F4320323499 must resolve to exactly one row in `openalex.funders.funders`; if not, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320323499;

## Step 2: Create BMEL Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bmel_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320323499  -- Bundesministerium für Ernährung und Landwirtschaft
),
g AS (
    SELECT
        TRIM(fkz) AS funder_award_id,
        NULLIF(TRIM(title), '') AS title,
        TRY_CAST(amount_eur AS DOUBLE) AS amount,
        NULLIF(TRIM(lps_text), '') AS scheme,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS start_d,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS end_d,
        CASE WHEN NULLIF(TRIM(recipient), '') IS NOT NULL AND recipient NOT LIKE 'Keine Anzeige%' THEN TRIM(recipient) END AS inst,
        NULLIF(TRIM(recipient_country_iso), '') AS inst_country,
        landing_page_url
    FROM openalex.awards.bmel_raw
    WHERE fkz IS NOT NULL AND TRIM(fkz) != '' AND ressort = 'BMLEH'
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
    g.title as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    g.funder_award_id,
    g.amount,
    'EUR' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    g.scheme as funder_scheme,
    'foerderkatalog_bmel' as provenance,
    g.start_d as start_date,
    g.end_d as end_date,
    YEAR(g.start_d) as start_year,
    YEAR(g.end_d) as end_year,
    -- No PI names in the Förderkatalog: institution-only lead (MPO/MEYS precedent);
    -- struct field order MUST match openalex_awards_raw.
    CASE WHEN g.inst IS NOT NULL THEN named_struct(
        'given_name', CAST(NULL AS STRING),
        'family_name', CAST(NULL AS STRING),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', g.inst,
            'country', g.inst_country,
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'foerderkatalog_bmel' AND priority = 532;

-- Insert into openalex_awards_raw with priority 532 (direct funder ingest; higher wins
-- under the 2026-06-20 DESC dedup, so it outranks the priority-0 acknowledgement shells).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    532 as priority
FROM openalex.awards.bmel_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.bmel_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.affiliation.name AS institution, landing_page_url
FROM openalex.awards.bmel_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.bmel_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 30;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.bmel_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency checks (no PI names published -> grantee institution and title).
SELECT lead_investigator.affiliation.name AS institution, COUNT(*) AS n
FROM openalex.awards.bmel_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.bmel_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator) as has_institution
FROM openalex.awards.bmel_awards;

In [ ]:
%sql
-- Shape check (2.1.1): FKZ only (6-12 alphanumerics, optional -suffix), never a blank id.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9A-Z]{6,12}(-[0-9A-Z]+)?$' THEN 1 ELSE 0 END) AS fkz_shaped,
    SUM(CASE WHEN NOT funder_award_id RLIKE '^[0-9A-Z]{6,12}(-[0-9A-Z]+)?$' THEN 1 ELSE 0 END) AS other,
    COUNT(*) AS total
FROM openalex.awards.bmel_awards;

In [ ]:
%sql
-- Citation stubs (priority < 3) under BMEL that collapse onto this ingest, and stubs under the
-- related rows (BLE, FNR, BMVEL) whose FKZ is in this ingest (these do NOT collapse).
SELECT c.funder_id, COUNT(*) as stub_rows,
       COUNT(t.id) as collapse_onto_bmel,
       COUNT(t2.k) as fkz_in_bmel_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.bmel_awards t ON t.id = c.id
LEFT JOIN (SELECT DISTINCT LOWER(funder_award_id) AS k FROM openalex.awards.bmel_awards) t2
       ON t2.k = LOWER(TRIM(c.funder_award_id))
WHERE c.funder_id IN (4320323499, 4320335545, 4320326662, 4320331007) AND c.priority < 3
GROUP BY 1 ORDER BY 1;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'foerderkatalog_bmel'
GROUP BY provenance, priority;